# Interactive Multi-Hazard Risk Dashboard

This notebook creates an interactive dashboard for exploring disaster risk data.
Designed for ministerial briefings and stakeholder presentations.

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import dash
from dash import dcc, html, Input, Output
import dash_bootstrap_components as dbc
import warnings
warnings.filterwarnings('ignore')

from config import PROCESSED_DATA_DIR, REPORTS_DIR

print("✓ Libraries loaded")

✓ Libraries loaded


## 1. Load Data

In [2]:
data_file = PROCESSED_DATA_DIR / 'disaster_events_sendai.csv'
risk_file = REPORTS_DIR / 'multi_hazard_risk_assessment.csv'

if data_file.exists():
    df = pd.read_csv(data_file)
    print(f"✓ Loaded {len(df)} disaster events")
else:
    print("Please run previous notebooks first")
    df = None

if risk_file.exists():
    risk_df = pd.read_csv(risk_file, index_col=0)
    print(f"✓ Loaded risk assessment for {len(risk_df)} locations")
else:
    risk_df = None

✓ Loaded 30 disaster events
✓ Loaded risk assessment for 25 locations


# Uncomment the line below to start the dashboard
# app.run(debug=True, port=8050, jupyter_mode='external')

In [3]:
if df is not None:
    app = dash.Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])
    
    app.layout = dbc.Container([
        dbc.Row([
            dbc.Col([
                html.H1("Multi-Hazard Risk Dashboard", className="text-center mb-4"),
                html.H5("Sendai Framework for Disaster Risk Reduction", 
                       className="text-center text-muted mb-4")
            ])
        ]),
        
        dbc.Row([
            dbc.Col([
                dbc.Card([
                    dbc.CardBody([
                        html.H4(f"{len(df):,}", className="text-primary"),
                        html.P("Total Events")
                    ])
                ])
            ], width=3),
            dbc.Col([
                dbc.Card([
                    dbc.CardBody([
                        html.H4(f"{df['total_deaths'].sum():,.0f}" if 'total_deaths' in df.columns else "N/A", 
                               className="text-danger"),
                        html.P("Total Deaths")
                    ])
                ])
            ], width=3),
            dbc.Col([
                dbc.Card([
                    dbc.CardBody([
                        html.H4(f"{df['total_affected'].sum()/1e6:.1f}M" if 'total_affected' in df.columns else "N/A", 
                               className="text-warning"),
                        html.P("People Affected")
                    ])
                ])
            ], width=3),
            dbc.Col([
                dbc.Card([
                    dbc.CardBody([
                        html.H4(f"${df['usdvalue'].sum()/1e9:.1f}B" if 'usdvalue' in df.columns else "N/A", 
                               className="text-success"),
                        html.P("Economic Losses")
                    ])
                ])
            ], width=3)
        ], className="mb-4"),
        
        dbc.Row([
            dbc.Col([
                dbc.Card([
                    dbc.CardHeader("Temporal Trends"),
                    dbc.CardBody([
                        dcc.Graph(id='temporal-graph')
                    ])
                ])
            ], width=6),
            dbc.Col([
                dbc.Card([
                    dbc.CardHeader("Hazard Distribution"),
                    dbc.CardBody([
                        dcc.Graph(id='hazard-pie')
                    ])
                ])
            ], width=6)
        ], className="mb-4"),
        
        dbc.Row([
            dbc.Col([
                dbc.Card([
                    dbc.CardHeader("Risk Assessment"),
                    dbc.CardBody([
                        dcc.Graph(id='risk-map')
                    ])
                ])
            ], width=12)
        ], className="mb-4"),
        
        dbc.Row([
            dbc.Col([
                dbc.Card([
                    dbc.CardHeader("Sendai Framework Indicators"),
                    dbc.CardBody([
                        dcc.Graph(id='sendai-indicators')
                    ])
                ])
            ], width=12)
        ])
    ], fluid=True)
    
    @app.callback(
        Output('temporal-graph', 'figure'),
        Input('temporal-graph', 'id')
    )
    def update_temporal(id):
        annual = df.groupby('year').size().reset_index(name='count')
        fig = px.line(annual, x='year', y='count', 
                     title='Annual Disaster Events',
                     labels={'count': 'Number of Events', 'year': 'Year'})
        fig.update_traces(line_color='#1f77b4', line_width=3)
        return fig
    
    @app.callback(
        Output('hazard-pie', 'figure'),
        Input('hazard-pie', 'id')
    )
    def update_hazard_pie(id):
        if 'hazardtype' in df.columns:
            hazard_counts = df['hazardtype'].value_counts().head(10)
            fig = px.pie(values=hazard_counts.values, names=hazard_counts.index,
                        title='Top 10 Hazard Types')
            return fig
        return go.Figure()
    
    @app.callback(
        Output('risk-map', 'figure'),
        Input('risk-map', 'id')
    )
    def update_risk_map(id):
        if risk_df is not None and 'risk_score' in risk_df.columns:
            top_risks = risk_df.nlargest(20, 'risk_score')
            fig = px.bar(top_risks, x=top_risks.index, y='risk_score',
                        title='Top 20 High-Risk Locations',
                        labels={'risk_score': 'Risk Score', 'index': 'Location'},
                        color='risk_score', color_continuous_scale='Reds')
            fig.update_layout(xaxis_tickangle=-45)
            return fig
        return go.Figure()
    
    @app.callback(
        Output('sendai-indicators', 'figure'),
        Input('sendai-indicators', 'id')
    )
    def update_sendai(id):
        sendai_cols = [col for col in df.columns if col.startswith('sendai_') and df[col].notna().sum() > 0]
        if sendai_cols:
            indicator_avgs = {col.replace('sendai_', '').replace('_', ' ').title(): df[col].mean() 
                            for col in sendai_cols[:6]}
            fig = go.Figure(data=[
                go.Bar(x=list(indicator_avgs.keys()), y=list(indicator_avgs.values()),
                      marker_color='steelblue')
            ])
            fig.update_layout(title='Average Sendai Framework Indicators',
                            xaxis_tickangle=-45,
                            yaxis_title='Average Value')
            return fig
        return go.Figure()
    
    print("\n✓ Dashboard created successfully!")
    print("\nTo run the dashboard, execute:")
    print("   app.run_server(debug=True, port=8050)")
    print("\nThen open: http://localhost:8050")


✓ Dashboard created successfully!

To run the dashboard, execute:
   app.run_server(debug=True, port=8050)

Then open: http://localhost:8050


## 3. Run Dashboard (Uncomment to Start)

In [6]:
# Uncomment the line below to start the dashboard
# app.run(debug=True, port=8050, jupyter_mode='external')

Dash app running on http://127.0.0.1:8050/


## 4. Alternative: Static Dashboard Export

In [5]:
if df is not None:
    from plotly.subplots import make_subplots
    
    fig = make_subplots(
        rows=3, cols=2,
        subplot_titles=('Annual Events', 'Hazard Distribution',
                       'Economic Losses', 'People Affected',
                       'Top Risk Locations', 'Sendai Indicators'),
        specs=[[{'type': 'scatter'}, {'type': 'pie'}],
               [{'type': 'bar'}, {'type': 'bar'}],
               [{'type': 'bar', 'colspan': 2}, None]],
        vertical_spacing=0.12,
        horizontal_spacing=0.1
    )
    
    annual = df.groupby('year').size().reset_index(name='count')
    fig.add_trace(
        go.Scatter(x=annual['year'], y=annual['count'], mode='lines+markers',
                  name='Events', line=dict(color='steelblue', width=3)),
        row=1, col=1
    )
    
    if 'hazardtype' in df.columns:
        hazard_counts = df['hazardtype'].value_counts().head(8)
        fig.add_trace(
            go.Pie(labels=hazard_counts.index, values=hazard_counts.values,
                  name='Hazards'),
            row=1, col=2
        )
    
    if 'usdvalue' in df.columns:
        annual_loss = df.groupby('year')['usdvalue'].sum().reset_index()
        annual_loss['usdvalue_millions'] = annual_loss['usdvalue'] / 1e6
        fig.add_trace(
            go.Bar(x=annual_loss['year'], y=annual_loss['usdvalue_millions'],
                  name='Economic Loss', marker_color='crimson'),
            row=2, col=1
        )
    
    if 'total_affected' in df.columns:
        annual_affected = df.groupby('year')['total_affected'].sum().reset_index()
        annual_affected['total_affected_thousands'] = annual_affected['total_affected'] / 1000
        fig.add_trace(
            go.Bar(x=annual_affected['year'], y=annual_affected['total_affected_thousands'],
                  name='Affected', marker_color='orange'),
            row=2, col=2
        )
    
    if risk_df is not None and 'risk_score' in risk_df.columns:
        top_risks = risk_df.nlargest(15, 'risk_score')
        fig.add_trace(
            go.Bar(x=top_risks.index, y=top_risks['risk_score'],
                  name='Risk Score', marker_color='darkred'),
            row=3, col=1
        )
    
    fig.update_layout(
        height=1200,
        showlegend=False,
        title_text="Multi-Hazard Risk Dashboard - Executive Summary",
        title_font_size=20
    )
    
    fig.update_xaxes(title_text="Year", row=1, col=1)
    fig.update_yaxes(title_text="Number of Events", row=1, col=1)
    fig.update_xaxes(title_text="Year", row=2, col=1)
    fig.update_yaxes(title_text="Loss (Million USD)", row=2, col=1)
    fig.update_xaxes(title_text="Year", row=2, col=2)
    fig.update_yaxes(title_text="Affected (Thousands)", row=2, col=2)
    fig.update_xaxes(title_text="Location", tickangle=-45, row=3, col=1)
    fig.update_yaxes(title_text="Risk Score", row=3, col=1)
    
    fig.show()
    
    from config import FIGURES_DIR
    fig.write_html(FIGURES_DIR / '06_executive_dashboard.html')
    print("\n✓ Static dashboard saved to outputs/figures/06_executive_dashboard.html")


✓ Static dashboard saved to outputs/figures/06_executive_dashboard.html


## 5. Dashboard Usage Guide

### For Presentations:
1. Open the HTML dashboard in a web browser
2. Use for ministerial briefings and stakeholder meetings
3. Interactive elements allow exploration during Q&A

### Key Features:
- **Summary Cards**: Quick overview of total impacts
- **Temporal Trends**: Event frequency over time
- **Hazard Distribution**: Most common disaster types
- **Risk Assessment**: Priority locations for intervention
- **Sendai Indicators**: Progress on international targets

### Customization:
- Modify colors in the code above
- Add filters for specific regions or time periods
- Include additional metrics as needed
- Export to PDF for printed reports